# Evaluation der Koreferenzauflösung

**Ordnerkontext:** liegt in `00_werkzeuge/`, liest aus
`02_quelldaten/pool_maerchen/` und `03_annotation_laufend/pronomen/`,
schreibt nach `05_ergebnisse/koref/`.

## Konfiguration

In [ ]:
from pathlib import Path

GOLD_PFAD  = (Path("..") / "04_konsolidiert" / "pronomen_gold_final.csv")
TEXTORDNER = Path("..") / "02_quelldaten" / "pool_maerchen"
AUSGABE    = Path("..") / "05_ergebnisse" / "koref"
ZWISCHEN   = AUSGABE / "aufgeloest"          
AUSGABE.mkdir(parents=True, exist_ok=True)
ZWISCHEN.mkdir(parents=True, exist_ok=True)

MODULPFAD = Path("..") / ".."

GOLD_PFAD, TEXTORDNER = GOLD_PFAD.resolve(), TEXTORDNER.resolve()
AUSGABE, ZWISCHEN = AUSGABE.resolve(), ZWISCHEN.resolve()
MODULPFAD = MODULPFAD.resolve()

SONDERWERTE = {"expletiv", "keine_figur", "kein_pronomen"}

IAA_QUOTE = 0.871
IAA_KAPPA = 0.858

REPS, SEED = 1000, 42
MIN_CLUSTER = 8   
MAX_ERSETZUNG = 4

UEBERSCHUSS_MAX = 2

NEU_BERECHNEN = False   
print(f"Gold: {GOLD_PFAD}")
print(f"Texte: {TEXTORDNER} | Zwischenspeicher: {ZWISCHEN}")
print(f"Agreement der Bearbeitenden: Quote {IAA_QUOTE}, Fleiss K {IAA_KAPPA}")
print()
print("Vorabfestlegung: Das Ergebnis dieses Laufs wird berichtet, wie es")
print("ausfaellt.")

Gold: /home/timc/Workspace/fairytale-personality-profiling/maerchen-goldstandard/04_konsolidiert/pronomen_gold_final.csv
Texte: /home/timc/Workspace/fairytale-personality-profiling/maerchen-goldstandard/02_quelldaten/pool_maerchen | Zwischenspeicher: /home/timc/Workspace/fairytale-personality-profiling/maerchen-goldstandard/05_ergebnisse/koref/aufgeloest
Agreement der Bearbeitenden: Quote 0.871, Fleiss K 0.858

Vorabfestlegung: Das Ergebnis dieses Laufs wird berichtet, wie es
ausfaellt.


## Lesepakete und Modul prüfen

In [12]:
import importlib, sys

for paket in ("pandas", "numpy"):
    if importlib.util.find_spec(paket) is None:
        raise ImportError(f"'{paket}' wird benoetigt: pip install {paket}")
print("  pandas, numpy vorhanden")

import os
from contextlib import contextmanager


@contextmanager
def im_pipelineverzeichnis():
    """Wechselt voruebergehend ins Pipeline-Projekt.

    Das Modul laedt beim Import `spacy.load("output/model-best")` ueber
    einen relativen Pfad. Aus 00_werkzeuge/ heraus zeigt der ins Leere,
    weshalb Import und Aufruf im Pipeline-Verzeichnis stattfinden.
    """
    vorher = Path.cwd()
    os.chdir(MODULPFAD)
    try:
        yield
    finally:
        os.chdir(vorher)


sys.path.insert(0, str(MODULPFAD))
try:
    with im_pipelineverzeichnis():
        from pipeline.coref_sliding_window import (
            resolve_coreferences_protokolliert)
    MODUL_DA = True
    print("  pipeline.coref_sliding_window geladen")
except Exception as fehler:
    MODUL_DA = False
    print(f"  Modul nicht geladen: {type(fehler).__name__}: {fehler}")
    print(f"  Erwartet wurde es unter {MODULPFAD}")
    print("  Ohne Modul koennen nur zwischengespeicherte Laeufe ausgewertet "
          "werden.")

vorhanden = sorted(ZWISCHEN.glob("*.txt"))
print(f"  Zwischengespeicherte Texte: {len(vorhanden)}")

  pandas, numpy vorhanden
  pipeline.coref_sliding_window geladen
  Zwischengespeicherte Texte: 20


## Schritt 1 — Golddatei einlesen und prüfen

In [ ]:
import pandas as pd

gold = pd.read_csv(GOLD_PFAD)
gold["token_index"] = gold["token_nr"] - 1         

def trifft(zeile):
    tokens = str(zeile["satz"]).split()
    i = zeile["token_index"]
    return 0 <= i < len(tokens) and tokens[i] == zeile["pronomen"]

fehl = gold[~gold.apply(trifft, axis=1)]
assert fehl.empty, f"{len(fehl)} Einheiten mit falscher Tokennummer"
assert gold["referent_gold"].notna().all(), "Einheiten ohne Zielwert"

gold["erwartet_figur"] = ~gold["referent_gold"].isin(SONDERWERTE)

print(f"Golddatei: {len(gold)} Einheiten aus "
      f"{gold['erzaehlung'].nunique()} Erzaehlungen")
print(f"  mit erwarteter Figur: {int(gold['erwartet_figur'].sum())}")
print(f"  Sonderwerte (keine Ersetzung erwartet): "
      f"{int((~gold['erwartet_figur']).sum())}")
print(f"  Tokennummer trifft das Pronomen: alle {len(gold)}")

Golddatei: 248 Einheiten aus 20 Erzaehlungen
  mit erwarteter Figur: 199
  Sonderwerte (keine Ersetzung erwartet): 49
  Tokennummer trifft das Pronomen: alle 248


## Schritt 2 — Quelltexte zuordnen

In [14]:
import re


def schluessel(name):
    """Vergleichsform eines Namens: nur Buchstaben und Ziffern.

    Die Dateinamen ersetzen Leerzeichen und Satzzeichen durch
    Unterstriche und tragen teils das Suffix -mod. Ein Vergleich ueber
    die Rohform schlaegt deshalb fehl, sobald der Titel ein Komma
    enthaelt. Verglichen wird daher allein die Buchstabenfolge.
    """
    roh = name.replace("-mod", "").lower()
    return re.sub(r"[^0-9a-z\u00e4\u00f6\u00fc\u00df]", "", roh)


texte = {schluessel(p.stem): p for p in sorted(TEXTORDNER.glob("*.txt"))}

zuordnung, fehlend = {}, []
for erz in sorted(gold["erzaehlung"].unique()):
    pfad = texte.get(schluessel(erz))
    if pfad is not None:
        zuordnung[erz] = pfad
    else:
        fehlend.append(erz)

print(f"Zugeordnet: {len(zuordnung)} von {gold['erzaehlung'].nunique()} "
      f"Erzaehlungen")
if fehlend:
    n = int(gold["erzaehlung"].isin(fehlend).sum())
    print(f"Ohne Quelltext ({len(fehlend)} Erzaehlungen, {n} Einheiten):")
    for e in fehlend:
        print(f"  {e}")
    print("Diese Einheiten bleiben unausgewertet.")
assert zuordnung, "kein einziger Quelltext gefunden"

Zugeordnet: 20 von 20 Erzaehlungen


## Schritt 3 — Koreferenzauflösung ausführen

In [15]:
import json, time

aufgeloest, protokolle = {}, {}
for erz, pfad in zuordnung.items():
    ziel_txt = ZWISCHEN / f"{pfad.stem}.txt"
    ziel_json = ZWISCHEN / f"{pfad.stem}_protokoll.json"

    if ziel_txt.exists() and not NEU_BERECHNEN:
        aufgeloest[erz] = ziel_txt.read_text(encoding="utf-8")
        if ziel_json.exists():
            protokolle[erz] = json.loads(ziel_json.read_text(encoding="utf-8"))
        print(f"  {erz[:44]:<44} aus dem Zwischenspeicher")
        continue

    if not MODUL_DA:
        print(f"  {erz[:44]:<44} uebersprungen (Modul fehlt)")
        continue

    beginn = time.time()
    text = pfad.read_text(encoding="utf-8")
    with im_pipelineverzeichnis():
        neuer_text, protokoll = resolve_coreferences_protokolliert(text)
    ziel_txt.write_text(neuer_text, encoding="utf-8")
    ziel_json.write_text(json.dumps(protokoll, ensure_ascii=False),
                         encoding="utf-8")
    aufgeloest[erz] = neuer_text
    protokolle[erz] = protokoll
    print(f"  {erz[:44]:<44} berechnet ({time.time()-beginn:.0f} s)")

print(f"\nAufgeloeste Texte: {len(aufgeloest)}")
if not aufgeloest:
    raise RuntimeError(
        "Kein aufgeloester Text verfuegbar. Entweder liegt das Modul nicht "
        f"unter {MODULPFAD} (Schritt 0b nennt den Grund), oder der "
        "Zwischenspeicher ist leer.")

  0_Alaeddin und die Wunderlampe               aus dem Zwischenspeicher
  0_Die Abenteuer Sindbads des Seefahrers      aus dem Zwischenspeicher
  0_Die Geschichte von Ali dem Kairenser und d aus dem Zwischenspeicher
  0_Dschudar und seine Brüder                  aus dem Zwischenspeicher
  0_Khalifah der Fischer von Bagdad            aus dem Zwischenspeicher
  1_Der Eisenhans                              aus dem Zwischenspeicher
  1_Der Räuberbräutigam                        aus dem Zwischenspeicher
  1_Die Gänsehirtin am Brunnen                 aus dem Zwischenspeicher
  1_Die drei Spinnerinnen                      aus dem Zwischenspeicher
  1_Märchen von einem, der auszog das Fürchten aus dem Zwischenspeicher
  2_Das Märchen vom Sandmann                   aus dem Zwischenspeicher
  2_Der Engel                                  aus dem Zwischenspeicher
  2_Des Kaisers neue Kleider                   aus dem Zwischenspeicher
  2_Die kleine Seejungfrau                     aus dem Zwischens

## Schritt 4 — Sätze und Token ausrichten

In [ ]:
import difflib, re

def worte(text):
    """Zerlegt in Woerter und trennt anhaengende Satzzeichen ab."""
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def anker(tokens):
    """Waehlt die laengsten Woerter des Satzes als Suchanker.

    Lange Woerter sind im Text selten und lokalisieren die Stelle
    zuverlaessiger als haeufige Funktionswoerter. Satzzeichen und kurze
    Woerter scheiden aus.
    """
    kandidaten = [w for w in tokens if len(w) >= 6 and w.isalpha()]
    return sorted(set(kandidaten), key=len, reverse=True)[:4]


def finde_stelle(goldtokens, textworte, mindest=0.55):
    """Sucht die Passage im aufgeloesten Text, die dem Goldsatz entspricht.

    Statt den Text in Saetze zu zerlegen, wird ueber Ankerwoerter grob
    lokalisiert und anschliessend ein Fenster um jede Fundstelle mit dem
    Goldsatz verglichen. Das ist robuster als eine Satzzerlegung, weil
    die Zerlegung des aufgeloesten Textes von der des Goldes abweichen
    kann. Rueckgabe: (Fenster als Wortliste, Aehnlichkeit).
    """
    laenge = len(goldtokens)
    stellen = set()
    for a in anker(goldtokens) or goldtokens[:2]:
        stellen.update(i for i, w in enumerate(textworte) if w == a)
    if not stellen:
        stellen = set(range(0, len(textworte), max(laenge // 2, 1)))

    beste, bester_wert = None, 0.0
    for pos in sorted(stellen):
        for versatz in range(-laenge, 1, max(laenge // 3, 1)):
            start = max(0, pos + versatz)
            fenster = textworte[start:start + laenge + 8]
            if not fenster:
                continue
            wert = difflib.SequenceMatcher(None, goldtokens, fenster,
                                           autojunk=False).ratio()
            if wert > bester_wert:
                beste, bester_wert = fenster, wert
    return (beste, bester_wert) if bester_wert >= mindest else (None, bester_wert)


def eingesetzt_an(goldtokens, index, aufgeloester_satz):
    """Liest, was im aufgeloesten Satz an der Stelle des Pronomens steht.

    Rueckgabe: der eingesetzte Wortlaut, oder None, wenn die Stelle nicht
    ausgerichtet werden konnte. Steht dort unveraendert das Pronomen,
    wird es selbst zurueckgegeben.
    """
    neu = (aufgeloester_satz if isinstance(aufgeloester_satz, list)
           else aufgeloester_satz.split())
    sm = difflib.SequenceMatcher(None, goldtokens, neu, autojunk=False)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if not (i1 <= index < i2):
            continue
        if tag == "equal":
            return goldtokens[index]
        if tag == "replace":                                                         
            stueck = neu[j1:j2]
            grenze = [i for i, w in enumerate(stueck)
                      if w in ".,;:!?›‹»«“„"]
            if grenze:
                vorne, hinten = stueck[:grenze[0]], stueck[grenze[-1] + 1:]
                stueck = hinten if len(hinten) > len(vorne) else vorne
            return " ".join(stueck[:MAX_ERSETZUNG])
        if tag == "delete":
            return ""
    return None

## Schritt 5 — Auswertung je Einheit

In [ ]:
ARTIKEL = {"der", "die", "das", "den", "dem", "des",
           "ein", "eine", "einen", "einem", "einer", "eines"}


def normalisiere(wortlaut):
    w = re.sub(r"[^\w\s\u00e4\u00f6\u00fc\u00df-]", " ", str(wortlaut).lower())
    teile = [t for t in w.split() if t]
    while teile and teile[0] in ARTIKEL:
        teile = teile[1:]
    return " ".join(teile)


#Wortlisten der aufgeloesten Texte einmal vorbereiten
textworte = {erz: worte(txt) for erz, txt in aufgeloest.items()}

def bezeichnet_dieselbe(eingesetzt, referent):
    a, b = normalisiere(eingesetzt).split(), normalisiere(referent).split()
    if not a or not b:
        return False
    if a == b:
        return True
    kurz, lang = (a, b) if len(a) <= len(b) else (b, a)
    if len(lang) - len(kurz) > UEBERSCHUSS_MAX:
        return False
    fuge = " ".join(kurz)
    return f" {fuge} " in f" {' '.join(lang)} "


ergebnisse = []
for _, z in gold.iterrows():
    if z["erzaehlung"] not in aufgeloest:
        ergebnisse.append(dict(z, klasse="ohne_quelltext", eingesetzt=None,
                               satzaehnlichkeit=None))
        continue

    tokens = str(z["satz"]).split()
    satz_neu, aehnlich = finde_stelle(tokens, textworte[z["erzaehlung"]])
    if satz_neu is None:
        ergebnisse.append(dict(z, klasse="satz_nicht_gefunden", eingesetzt=None,
                               satzaehnlichkeit=round(aehnlich, 3)))
        continue

    wortlaut = eingesetzt_an(tokens, int(z["token_index"]), satz_neu)
    if wortlaut is None:
        ergebnisse.append(dict(z, klasse="nicht_ausgerichtet", eingesetzt=None,
                               satzaehnlichkeit=round(aehnlich, 3)))
        continue

    unveraendert = normalisiere(wortlaut) == normalisiere(z["pronomen"])
    if z["erwartet_figur"]:
        if unveraendert or not wortlaut.strip():
            klasse = "nicht_ersetzt"
        else:
            klasse = ("korrekt" if bezeichnet_dieselbe(wortlaut,
                                                       z["referent_gold"])
                      else "falsche_figur")
    else:
        klasse = "unveraendert_korrekt" if unveraendert else "faelschlich_ersetzt"

    ergebnisse.append(dict(z, klasse=klasse, eingesetzt=wortlaut,
                           satzaehnlichkeit=round(aehnlich, 3)))

erg = pd.DataFrame(ergebnisse)

#Nachweis, wie viele Treffer nicht auf vollstaendiger Uebereinstimmung
#beruhen, sondern auf einer Teiluebereinstimmung
teil = sum(1 for _, r in erg.iterrows()
           if r["klasse"] == "korrekt"
           and normalisiere(r["eingesetzt"]) != normalisiere(r["referent_gold"]))

from collections import Counter
print("Ergebnisklassen:")
for k, n in Counter(erg["klasse"]).most_common():
    print(f"  {k:<22}{n:>5}  {n/len(erg):>6.1%}")
print()
print(f"Davon als korrekt gewertet ueber eine Teiluebereinstimmung: {teil}")
print("  (etwa 'Prinzessin Bedrulbudur' fuer 'Prinzessin')")

Ergebnisklassen:
  korrekt                  95   38.3%
  falsche_figur            52   21.0%
  unveraendert_korrekt     45   18.1%
  nicht_ersetzt            44   17.7%
  satz_nicht_gefunden      10    4.0%
  faelschlich_ersetzt       2    0.8%

Davon als korrekt gewertet ueber eine Teiluebereinstimmung: 21
  (etwa 'Prinzessin Bedrulbudur' fuer 'Prinzessin')


## Schritt 6 — Kennzahlen

In [18]:
import numpy as np

rng = np.random.default_rng(SEED)
bewertbar = erg[~erg["klasse"].isin(
    ["ohne_quelltext", "satz_nicht_gefunden", "nicht_ausgerichtet"])]
mit_figur = bewertbar[bewertbar["erwartet_figur"]]
ohne_figur = bewertbar[~bewertbar["erwartet_figur"]]

def quote(teil, treffer):
    return (teil["klasse"] == treffer).mean() if len(teil) else float("nan")

q_auf = quote(mit_figur, "korrekt")
q_son = quote(ohne_figur, "unveraendert_korrekt")

def cluster_ci(teil, treffer):
    erz = sorted(teil["erzaehlung"].unique())
    if len(erz) < 2:
        return float("nan"), float("nan")
    je = {e: teil[teil["erzaehlung"] == e] for e in erz}
    idx = rng.integers(0, len(erz), size=(REPS, len(erz)))
    werte = []
    for zeile in idx:
        gezogen = pd.concat([je[erz[i]] for i in zeile])
        werte.append((gezogen["klasse"] == treffer).mean())
    return tuple(float(x) for x in np.percentile(werte, [2.5, 97.5]))

lo, hi = cluster_ci(mit_figur, "korrekt")
k = mit_figur["erzaehlung"].nunique()

print(f"Auswertbare Einheiten: {len(bewertbar)} von {len(erg)}")
print()
print(f"Aufloesungsquote (erwartete Figur, n = {len(mit_figur)}): {q_auf:.3f}")
print(f"  Cluster-Bootstrap ueber {k} Erzaehlungen: [{lo:.3f}, {hi:.3f}]")
if k < MIN_CLUSTER:
    print(f"  HINWEIS: weniger als {MIN_CLUSTER} Erzaehlungen; das Intervall")
    print("  faellt zu optimistisch aus und ist als Richtungsangabe zu lesen.")
print()
print(f"Fehlerklassen bei erwarteter Figur:")
for k_ in ("falsche_figur", "nicht_ersetzt"):
    n = int((mit_figur["klasse"] == k_).sum())
    print(f"  {k_:<18}{n:>5}  {n/max(len(mit_figur),1):>6.1%}")
print()
print(f"Sonderwerte (keine Ersetzung erwartet, n = {len(ohne_figur)}): "
      f"{q_son:.3f} korrekt unveraendert")
n_f = int((ohne_figur["klasse"] == "faelschlich_ersetzt").sum())
print(f"  faelschlich ersetzt: {n_f}")
print()
print(f"Zum Vergleich die Uebereinstimmung der Bearbeitenden: "
      f"Quote {IAA_QUOTE:.3f}, Fleiss K {IAA_KAPPA:.3f}")
if q_auf == q_auf and q_auf > IAA_QUOTE:
    print("  HINWEIS: Die Quote liegt ueber der menschlichen "
          "Uebereinstimmung.")

Auswertbare Einheiten: 238 von 248

Aufloesungsquote (erwartete Figur, n = 191): 0.497
  Cluster-Bootstrap ueber 20 Erzaehlungen: [0.356, 0.638]

Fehlerklassen bei erwarteter Figur:
  falsche_figur        52   27.2%
  nicht_ersetzt        44   23.0%

Sonderwerte (keine Ersetzung erwartet, n = 47): 0.957 korrekt unveraendert
  faelschlich ersetzt: 2

Zum Vergleich die Uebereinstimmung der Bearbeitenden: Quote 0.871, Fleiss K 0.858


## Schritt 7 — Aufschlüsselung und Fehlerliste

In [19]:
for spalte in ("typ", "sammlung"):
    print(f"Nach {spalte}:")
    for wert, teil in mit_figur.groupby(spalte):
        print(f"  {str(wert):<16}{(teil['klasse']=='korrekt').mean():.3f}"
              f"  (n={len(teil)})")
    print()

print("Erzaehlungen mit der niedrigsten Quote:")
je_erz = (mit_figur.groupby("erzaehlung")
          .apply(lambda t: pd.Series({"quote": (t["klasse"]=="korrekt").mean(),
                                      "n": len(t)}), include_groups=False)
          .sort_values("quote"))
for erz, r in je_erz.head(5).iterrows():
    print(f"  {erz[:46]:<46}{r['quote']:.3f}  (n={int(r['n'])})")

fehler = bewertbar[bewertbar["klasse"].isin(
    ["falsche_figur", "nicht_ersetzt", "faelschlich_ersetzt"])]
spalten = ["klasse", "erzaehlung", "sammlung", "satzindex", "token_nr",
           "pronomen", "typ", "referent_gold", "eingesetzt", "satz_markiert"]
fehler[spalten].to_csv(AUSGABE / "fehlerliste_koref.csv", index=False,
                       encoding="utf-8")
print(f"\nFehlerliste: {len(fehler)} Eintraege -> "
      f"{AUSGABE / 'fehlerliste_koref.csv'}")

Nach typ:
  demonstrativ    0.154  (n=13)
  personal        0.522  (n=178)

Nach sammlung:
  Andersen        0.514  (n=35)
  Arabisch        0.491  (n=53)
  Grimm           0.420  (n=50)
  Japanisch       0.566  (n=53)

Erzaehlungen mit der niedrigsten Quote:
  1_Die Gänsehirtin am Brunnen                  0.000  (n=6)
  0_Khalifah der Fischer von Bagdad             0.000  (n=8)
  2_Das Märchen vom Sandmann                    0.000  (n=1)
  0_Dschudar und seine Brüder                   0.154  (n=13)
  2_Die roten Schuhe                            0.182  (n=11)

Fehlerliste: 98 Eintraege -> /home/timc/Workspace/fairytale-personality-profiling/maerchen-goldstandard/05_ergebnisse/koref/fehlerliste_koref.csv


## Schritt 8 — Ergebnis sichern

In [20]:
from datetime import datetime

bericht = {
    "zeitpunkt": datetime.now().isoformat(timespec="minutes"),
    "ebene": "koreferenz",
    "gold": str(GOLD_PFAD),
    "einheiten_gesamt": int(len(erg)),
    "einheiten_auswertbar": int(len(bewertbar)),
    "einheiten_mit_figur": int(len(mit_figur)),
    "einheiten_sonderwerte": int(len(ohne_figur)),
    "erzaehlungen": int(k),
    "aufloesungsquote": float(q_auf),
    "cluster_bootstrap_95": [lo, hi],
    "sonderwerte_korrekt": float(q_son),
    "klassen": {k_: int(v) for k_, v in Counter(erg["klasse"]).items()},
    "nach_typ": {str(w): float((t["klasse"]=="korrekt").mean())
                 for w, t in mit_figur.groupby("typ")},
    "nach_sammlung": {str(w): float((t["klasse"]=="korrekt").mean())
                      for w, t in mit_figur.groupby("sammlung")},
    "agreement_bearbeitende": {"quote": IAA_QUOTE, "fleiss_kappa": IAA_KAPPA},
}
ziel = AUSGABE / "koref_eval.json"
ziel.write_text(json.dumps(bericht, ensure_ascii=False, indent=2),
                encoding="utf-8")
print(f"Geschrieben: {ziel}")
print()
print(f"Aufloesungsquote {q_auf:.3f} auf {len(mit_figur)} Einheiten "
      f"aus {k} Erzaehlungen.")
print("Die Quote ist bedingt auf die Trefferquote der maschinellen")
print("Pronomenextraktion zu lesen: uebersehene Pronomen erscheinen nicht")
print("in der Golddatei und bleiben deshalb ungemessen.")

Geschrieben: /home/timc/Workspace/fairytale-personality-profiling/maerchen-goldstandard/05_ergebnisse/koref/koref_eval.json

Aufloesungsquote 0.497 auf 191 Einheiten aus 20 Erzaehlungen.
Die Quote ist bedingt auf die Trefferquote der maschinellen
Pronomenextraktion zu lesen: uebersehene Pronomen erscheinen nicht
in der Golddatei und bleiben deshalb ungemessen.
